In [1]:
import numpy as np
import torch
import torch.nn as nn

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

In [2]:
data = np.load("pems04.npz")

traffic = data["data"][:,:,2]

print(traffic.shape)

(16992, 307)


In [3]:
scaler = MinMaxScaler()

traffic_scaled = scaler.fit_transform(
    traffic
)

print(
    traffic_scaled.shape
)

(16992, 307)


In [4]:
X = []
y = []

sequence_length = 12

for i in range(
    len(traffic_scaled)
    -
    sequence_length
):

    X.append(
        traffic_scaled[
            i:i+sequence_length
        ]
    )

    y.append(
        traffic_scaled[
            i+sequence_length
        ]
    )

X = np.array(X)
y = np.array(y)

print(X.shape)
print(y.shape)

(16980, 12, 307)
(16980, 307)


In [5]:
split = int(
    len(X) * 0.8
)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

print(X_train.shape)
print(X_test.shape)

(13584, 12, 307)
(3396, 12, 307)


In [6]:
X_train = torch.tensor(
    X_train,
    dtype=torch.float32
)

X_test = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_train = torch.tensor(
    y_train,
    dtype=torch.float32
)

y_test = torch.tensor(
    y_test,
    dtype=torch.float32
)

In [7]:
train_dataset = TensorDataset(
    X_train,
    y_train
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

In [8]:
import pickle
import numpy as np

with open("adj_mx_bay.pkl", "rb") as f:
    sensor_ids, sensor_id_to_ind, adj_mx = pickle.load(
        f,
        encoding="latin1"
    )

print(adj_mx.shape)

print(adj_mx.min())
print(adj_mx.max())

(325, 325)
0.0
1.0


In [9]:
A = adj_mx

A = A + np.eye(A.shape[0])

D = np.diag(
    np.power(
        A.sum(axis=1),
        -0.5
    )
)

A_hat = D @ A @ D

print(A_hat.shape)

(325, 325)


In [10]:
import torch

A_hat = torch.tensor(
    A_hat,
    dtype=torch.float32
)

print(A_hat.shape)

torch.Size([325, 325])


In [11]:
class GraphConv(nn.Module):

    def __init__(
        self,
        num_nodes,
        in_features,
        out_features
    ):

        super().__init__()

        self.node_embeddings = nn.Parameter(
            torch.randn(
                num_nodes,
                16
            )
        )

        self.linear = nn.Linear(
            in_features,
            out_features
        )

    def forward(self,x):

        adj = torch.softmax(
            torch.relu(
                self.node_embeddings
                @
                self.node_embeddings.T
            ),
            dim=1
        )

        x = torch.einsum(
            "ij,btj->bti",
            adj,
            x
        )

        x = self.linear(x)

        return torch.relu(x)

In [12]:
class GCNLSTM(nn.Module):

    def __init__(self):

        super().__init__()

        self.gcn = GraphConv(
            num_nodes=307,
            in_features=307,
            out_features=307
        )

        self.lstm = nn.LSTM(
            input_size=307,
            hidden_size=64,
            batch_first=True
        )

        self.fc = nn.Linear(
            64,
            307
        )

    def forward(self,x):

        x = self.gcn(x)

        output, (hidden, cell) = self.lstm(x)

        hidden = hidden[-1]

        x = self.fc(hidden)

        return x

In [13]:
model = GCNLSTM()

X_batch, y_batch = next(iter(train_loader))

pred = model(X_batch)

print(pred.shape)
print(y_batch.shape)

torch.Size([64, 307])
torch.Size([64, 307])


In [14]:
model = GCNLSTM()

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

import time

train_start = time.time()

epochs = 30

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        pred = model(X_batch)

        loss = criterion(
            pred,
            y_batch
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch+1}/{epochs}",
        f"Loss: {total_loss/len(train_loader):.6f}"
    )

Epoch 1/30 Loss: 0.046472
Epoch 2/30 Loss: 0.009014
Epoch 3/30 Loss: 0.007707
Epoch 4/30 Loss: 0.007218
Epoch 5/30 Loss: 0.006666
Epoch 6/30 Loss: 0.006229
Epoch 7/30 Loss: 0.005942
Epoch 8/30 Loss: 0.005694
Epoch 9/30 Loss: 0.005478
Epoch 10/30 Loss: 0.005299
Epoch 11/30 Loss: 0.005163
Epoch 12/30 Loss: 0.005037
Epoch 13/30 Loss: 0.004925
Epoch 14/30 Loss: 0.004811
Epoch 15/30 Loss: 0.004715
Epoch 16/30 Loss: 0.004606
Epoch 17/30 Loss: 0.004497
Epoch 18/30 Loss: 0.004416
Epoch 19/30 Loss: 0.004319
Epoch 20/30 Loss: 0.004258
Epoch 21/30 Loss: 0.004181
Epoch 22/30 Loss: 0.004111
Epoch 23/30 Loss: 0.004031
Epoch 24/30 Loss: 0.003977
Epoch 25/30 Loss: 0.003908
Epoch 26/30 Loss: 0.003849
Epoch 27/30 Loss: 0.003792
Epoch 28/30 Loss: 0.003739
Epoch 29/30 Loss: 0.003689
Epoch 30/30 Loss: 0.003641


In [15]:
train_time= time.time() - train_start

print("Time Taken:", train_time)

Time Taken: 214.7327561378479


In [16]:
torch.save(
    model.state_dict(),
    "STGCN-PEMS04.pth"
)

In [17]:
model.eval()

infer_start = time.time()

with torch.no_grad():

    predictions = model(
        X_test
    )

predictions = predictions.numpy()

infer_time = time.time() - infer_start
print("Infer Time:", infer_time)

true_values = y_test.numpy()

mae = mean_absolute_error(
    true_values,
    predictions
)

rmse = np.sqrt(
    mean_squared_error(
        true_values,
        predictions
    )
)

print("MAE:", mae)
print("RMSE:", rmse)

Infer Time: 0.21103119850158691
MAE: 0.040365484
RMSE: 0.070305824


In [18]:
from sklearn.metrics import r2_score

mape = np.mean(
    np.abs((true_values - predictions) /
           np.maximum(np.abs(true_values), 1e-6))
) * 100

r2 = r2_score(
    true_values.flatten(),
    predictions.flatten()
)
print("MAPE:", mape)
print("R2:", r2)

MAPE: 6439.141845703125
R2: 0.7649515966902392


In [19]:
params = sum(
    p.numel()
    for p in model.parameters()
)

print("Parameters:", params)

Parameters: 214911
